### ESPCN example

ℹ `xpi-nn-train` is opt for **image classification** benchmarking, here we give a more complex image super-resolution example :)

⚪ Model

In [1]:
from xpi_nn_train.models.espcn import ESPCN

# for RGB images, upscale x2
model = ESPCN(upscale_factor=2)
model.requires_grad_(True)
print(model)
print('param_cnt:', sum(p.numel() for p in model.parameters() if p.requires_grad))

ESPCN(
  (feature_maps): Sequential(
    (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): Tanh()
    (2): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): Tanh()
    (4): Conv2d(32, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (sub_pixel): PixelShuffle(upscale_factor=2)
)
param_cnt: 26796


⚪ Dataset

In [2]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch import Tensor

from xpi_nn_train.datasets.torchvision import get_dataset
from xpi_nn_train.trainers import get_trainer_args

class CIFAR10_SR:

  def __init__(self, split='train'):
    self.dataset = get_dataset('CIFAR10', split)

  def __len__(self) -> int:
    return len(self.dataset)

  def __getitem__(self, index:int) -> tuple[Tensor, Tensor]:
    img, lbl = self.dataset[index]
    img_low = F.interpolate(img.unsqueeze(0), scale_factor=0.5).squeeze(0)
    return (img_low, img)

args = get_trainer_args()
args.epochs = 1
args.learning_rate = 2e-3
print(args)

train_loader = DataLoader(CIFAR10_SR('train'), batch_size=args.batch_size, num_workers=args.num_workers)
test_loader = DataLoader(CIFAR10_SR('val'), batch_size=args.batch_size, num_workers=args.num_workers)
print(f'train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')
print(f'test: {len(test_loader.dataset)} samples, {len(test_loader)} batches')

Namespace(backend='torch', num_classes=10, epochs=1, batch_size=32, grad_acc=4, learning_rate=0.002, lora=0, num_workers=0, load=None, seed=114514)
DATA_ROOT: C:\Workspace\xpi-nn-train\DATA_ROOT
train: 50000 samples, 1563 batches
test: 10000 samples, 313 batches


⚪ Train

Need copy & modify the whole `run_train_native()` training loop :(
Adapt your data batch, loss function and metrics statistics

In [ ]:
from time import time

import torch
import torch.nn.functional as F
from torch.optim import Adam

from xpi_nn_train.utils import seed_everything, ValueWindow

torch.set_num_threads(torch.get_num_interop_threads())
dtype = torch.float32  # fp16/bf16 on most SoC is slow
seed_everything(args.seed)

model = model.to(dtype)
optim = Adam(model.parameters(), lr=args.learning_rate)

vw = ValueWindow(100)
step = 1
ts_init = time()
for epoch in range(1, 1 + args.epochs):
  ts_epoch = time()

  ''' Train '''
  cnt, sum = 0, 0
  model.train()
  optim.zero_grad()
  for x, y in train_loader:
    ts_step = time()
    x = x.to(dtype)
    y = y.to(dtype)
    x_rec = model(x)
    loss = F.mse_loss(x_rec, y)
    if args.grad_acc > 1:
      loss = loss / args.grad_acc
    loss.backward()
    if step % args.grad_acc == 0:
      optim.step()
      optim.zero_grad()
    vw.append(time() - ts_step)

    with torch.inference_mode():
      sum += loss.item()
      cnt += x.size(0)

    step += 1
    if step % vw.nlen == 0:
      print(f'[Step {step}] loss: {loss.item():.5f}, mse: {sum / cnt} ({1 / vw.mean:.5f} it/s)')

  ''' Test '''
  with torch.inference_mode():
    cnt, sum = 0, 0
    model.eval()
    for x, y in test_loader:
      x = x.to(dtype)
      y = y.to(dtype)

      x_rec = model(x)
      loss = F.mse_loss(x_rec, y)
      sum += loss.item()
      cnt += x.size(0)

  print(f'[Epoch {epoch}] Test MSE: {sum / cnt} ({time() - ts_epoch:.5f}s)\n')

ts_done = time() - ts_init
print(f'Done: {ts_done:.5f}s')

[Step 100] loss: 0.25689, mse: 0.008928978665158002 (156.46360 it/s)
[Step 200] loss: 0.28274, mse: 0.00859356215972472 (161.27931 it/s)
[Step 300] loss: 0.24350, mse: 0.008338705407623763 (167.33188 it/s)
[Step 400] loss: 0.23897, mse: 0.008171677105082082 (170.51825 it/s)
[Step 500] loss: 0.27668, mse: 0.008107856056001598 (169.57138 it/s)
[Step 600] loss: 0.27447, mse: 0.008046590430795749 (169.17448 it/s)
[Step 700] loss: 0.24534, mse: 0.008000503197573498 (165.16981 it/s)
[Step 800] loss: 0.21520, mse: 0.007968693227391778 (159.43408 it/s)
[Step 900] loss: 0.26846, mse: 0.007943524830395541 (160.79137 it/s)
[Step 1000] loss: 0.23020, mse: 0.007916297928129023 (170.54370 it/s)
[Step 1100] loss: 0.25831, mse: 0.007899465688501736 (165.21392 it/s)
[Step 1200] loss: 0.23231, mse: 0.00787647987783477 (165.90808 it/s)
[Step 1300] loss: 0.23796, mse: 0.007854468391002303 (166.05173 it/s)
[Step 1400] loss: 0.24531, mse: 0.007838754800865273 (164.77892 it/s)
[Step 1500] loss: 0.23514, mse: